# Ingest Price History and Indices (nepse-api)

This notebook populates both:
- `price_history` from company historical OHLCV
- `indices` from NEPSE index history

Canonical `index_name` values used in this project (for benchmark Q11):
- `NEPSE`
- `Banking`
- `Hydropower`

In [1]:
from __future__ import annotations

import asyncio
from datetime import date, datetime
import csv
import io
import json
from pathlib import Path
import sqlite3
import sys
import time
from typing import Any
from urllib.error import HTTPError, URLError
from urllib.parse import urlencode
from urllib.request import Request, urlopen

from tqdm.auto import tqdm

from nepse import Client

# Make project imports work both locally and in Colab.
PROJECT_ROOT = Path.cwd()
if (PROJECT_ROOT / "nepse_analyst").exists() is False:
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from nepse_analyst.database import create_database
from nepse_analyst.config import DB_PATH

create_database()
print(f"DB_PATH: {DB_PATH}")

DB_PATH: /home/meutsabdahal/Desktop/nepse-analyst/data/processed/nepse.db


/home/meutsabdahal/Desktop/nepse-analyst/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
BATCH_SIZE = 500
SYMBOL_SLEEP_SECONDS = 1
HTTP_TIMEOUT_SECONDS = 8
MAX_HTTP_RETRIES = 1
MAX_ERROR_LOGS = 15

FALLBACK_API_BASE = "https://nepseapi.surajrimal.dev"
GITHUB_COMPANY_WISE_URL = (
    "https://api.github.com/repos/Aabishkar2/nepse-data/contents/data/company-wise"
)
GITHUB_INDEX_DAILY_URL = "https://api.github.com/repos/socrateai-official/nepse-open-data/contents/ohlc_index"

_GITHUB_SYMBOL_FILE_MAP: dict[str, str] | None = None
_GITHUB_INDEX_FILE_URLS: list[str] | None = None

CANONICAL_INDEX_NAMES = {
    "NEPSE": ["nepse", "nepse index", "all share"],
    "Banking": ["banking", "bank"],
    "Hydropower": ["hydropower", "hydro power", "hydro"],
}


def _to_dict(obj: Any) -> dict[str, Any]:
    if obj is None:
        return {}
    if isinstance(obj, dict):
        return obj
    if hasattr(obj, "dict") and callable(obj.dict):
        try:
            return obj.dict()
        except Exception:
            pass
    if hasattr(obj, "__dict__"):
        return dict(vars(obj))
    return {}


def _first_non_null(obj: Any, keys: list[str], default: Any = None) -> Any:
    if obj is None:
        return default

    if isinstance(obj, dict):
        for k in keys:
            if k in obj and obj[k] is not None:
                return obj[k]
        return default

    for k in keys:
        if hasattr(obj, k):
            value = getattr(obj, k)
            if value is not None:
                return value

    obj_dict = _to_dict(obj)
    for k in keys:
        if k in obj_dict and obj_dict[k] is not None:
            return obj_dict[k]

    return default


def _to_iso_date(value: Any) -> str | None:
    if value is None:
        return None
    if isinstance(value, datetime):
        return value.date().isoformat()
    if isinstance(value, date):
        return value.isoformat()

    text = str(value).strip()
    if not text:
        return None

    for parser in (datetime.fromisoformat,):
        try:
            return parser(text.replace("Z", "+00:00")).date().isoformat()
        except Exception:
            pass

    candidates = ["%Y-%m-%d", "%d-%m-%Y", "%Y/%m/%d", "%d/%m/%Y"]
    for fmt in candidates:
        try:
            return datetime.strptime(text, fmt).date().isoformat()
        except ValueError:
            continue

    if len(text) >= 10 and text[4] == "-" and text[7] == "-":
        return text[:10]

    return None


def _to_float(value: Any) -> float | None:
    if value is None:
        return None
    if isinstance(value, bool):
        return float(value)
    text = str(value).replace(",", "").strip()
    if text == "":
        return None
    try:
        return float(text)
    except ValueError:
        return None


def _to_int(value: Any) -> int | None:
    number = _to_float(value)
    if number is None:
        return None
    return int(number)


def _normalize_rows(payload: Any) -> list[Any]:
    if payload is None:
        return []
    if isinstance(payload, list):
        return payload
    if isinstance(payload, tuple):
        return list(payload)
    if isinstance(payload, dict):
        for key in (
            "data",
            "content",
            "results",
            "result",
            "items",
            "history",
            "response",
        ):
            if key in payload and isinstance(payload[key], list):
                return payload[key]
        return [payload]

    payload_dict = _to_dict(payload)
    for key in ("data", "content", "results", "result", "items", "history", "response"):
        candidate = payload_dict.get(key)
        if isinstance(candidate, list):
            return candidate

    return [payload]


def _canonical_index_name(raw_name: Any) -> str | None:
    if raw_name is None:
        return None
    text = str(raw_name).strip().lower().replace("%20", " ")

    if "bank" in text:
        return "Banking"
    if "hydro" in text:
        return "Hydropower"
    if "nepse" in text or "all share" in text:
        return "NEPSE"
    return None


def _http_get_json(
    path_or_url: str, params: dict[str, Any] | None = None, absolute_url: bool = False
) -> Any:
    query = f"?{urlencode(params)}" if params else ""
    url = (
        f"{path_or_url}{query}"
        if absolute_url
        else f"{FALLBACK_API_BASE}{path_or_url}{query}"
    )

    last_exc: Exception | None = None
    for attempt in range(1, MAX_HTTP_RETRIES + 1):
        try:
            req = Request(
                url,
                headers={
                    "User-Agent": "Mozilla/5.0",
                    "Accept": "application/json",
                },
            )
            with urlopen(req, timeout=HTTP_TIMEOUT_SECONDS) as resp:
                body = resp.read().decode("utf-8")
                return json.loads(body)
        except (HTTPError, URLError, TimeoutError, json.JSONDecodeError) as exc:
            last_exc = exc
            if attempt < MAX_HTTP_RETRIES:
                time.sleep(1)
                continue
            break

    raise RuntimeError(f"HTTP fetch failed for {url}: {last_exc}")


def _http_get_text(url: str) -> str:
    last_exc: Exception | None = None
    for attempt in range(1, MAX_HTTP_RETRIES + 1):
        try:
            req = Request(
                url,
                headers={
                    "User-Agent": "Mozilla/5.0",
                    "Accept": "text/csv,text/plain,*/*",
                },
            )
            with urlopen(req, timeout=HTTP_TIMEOUT_SECONDS) as resp:
                return resp.read().decode("utf-8")
        except (HTTPError, URLError, TimeoutError) as exc:
            last_exc = exc
            if attempt < MAX_HTTP_RETRIES:
                time.sleep(1)
                continue
            break

    raise RuntimeError(f"HTTP text fetch failed for {url}: {last_exc}")


def _load_github_symbol_file_map() -> dict[str, str]:
    global _GITHUB_SYMBOL_FILE_MAP
    if _GITHUB_SYMBOL_FILE_MAP is not None:
        return _GITHUB_SYMBOL_FILE_MAP

    payload = _http_get_json(GITHUB_COMPANY_WISE_URL, absolute_url=True)
    items = _normalize_rows(payload)

    mapping: dict[str, str] = {}
    for item in items:
        name = str(_first_non_null(item, ["name"], "")).strip()
        if not name.lower().endswith(".csv"):
            continue
        symbol = name[:-4].upper()
        download_url = _first_non_null(item, ["download_url", "downloadUrl"])
        if not download_url:
            continue
        mapping[symbol] = str(download_url)

    _GITHUB_SYMBOL_FILE_MAP = mapping
    return mapping


def _get_history_rows_github(symbol: str) -> list[dict[str, Any]]:
    file_map = _load_github_symbol_file_map()
    csv_url = file_map.get(symbol.upper())
    if not csv_url:
        return []

    text = _http_get_text(csv_url)
    reader = csv.DictReader(io.StringIO(text))
    rows: list[dict[str, Any]] = []
    for row in reader:
        rows.append(
            {
                "date": row.get("published_date"),
                "open": row.get("open"),
                "high": row.get("high"),
                "low": row.get("low"),
                "close": row.get("close"),
                "volume": row.get("traded_quantity"),
                "turnover": row.get("traded_amount"),
            }
        )
    return rows


def _load_github_index_file_urls() -> list[str]:
    global _GITHUB_INDEX_FILE_URLS
    if _GITHUB_INDEX_FILE_URLS is not None:
        return _GITHUB_INDEX_FILE_URLS

    payload = _http_get_json(GITHUB_INDEX_DAILY_URL, absolute_url=True)
    items = _normalize_rows(payload)
    urls: list[str] = []
    for item in items:
        name = str(_first_non_null(item, ["name"], "")).strip()
        if not name.startswith("adj_") or not name.endswith(".csv"):
            continue
        download_url = _first_non_null(item, ["download_url", "downloadUrl"])
        if download_url:
            urls.append(str(download_url))

    # Newest files first so we can stop early once enough days are collected.
    urls.sort(reverse=True)
    _GITHUB_INDEX_FILE_URLS = urls
    return urls


def _is_equity_symbol(symbol: str) -> bool:
    # Keep listed equity symbols and skip debentures/bonds/promoter variants.
    text = symbol.strip().upper()
    if not text:
        return False
    if "%" in text:
        return False
    if any(ch.isdigit() for ch in text):
        return False
    if text.endswith("P"):
        return False
    return text.isalpha()

In [3]:
PRICE_INSERT_SQL = """
INSERT INTO price_history (
    symbol, trade_date, open_price, high_price, low_price, close_price, volume, turnover
) VALUES (?, ?, ?, ?, ?, ?, ?, ?)
"""


async def _get_history_rows_nepse_api(client: Client, symbol: str) -> list[Any]:
    company = await client.security_client.get_company(symbol=symbol)
    company_id = _first_non_null(company, ["security_id", "id"])
    if company_id is None:
        raise ValueError(f"Missing security_id for symbol {symbol}")
    history = await client.security_client.get_company_history(company_id=company_id)
    return _normalize_rows(history)


def _get_history_rows_fallback_api(symbol: str) -> list[Any]:
    payload = _http_get_json("/PriceVolumeHistory", params={"symbol": symbol})
    return _normalize_rows(payload)


async def _preflight_api(client: Client) -> tuple[bool, bool, bool]:
    nepse_ok = False
    fallback_ok = False
    github_ok = False

    try:
        await _get_history_rows_nepse_api(client, "ADBL")
        nepse_ok = True
    except Exception:
        nepse_ok = False

    try:
        fallback_probe_rows = _get_history_rows_fallback_api("ADBL")
        fallback_ok = len(fallback_probe_rows) > 0
    except Exception:
        fallback_ok = False

    try:
        file_map = _load_github_symbol_file_map()
        github_ok = len(file_map) > 0
    except Exception:
        github_ok = False

    return nepse_ok, fallback_ok, github_ok


async def ingest_price_history() -> None:
    conn = sqlite3.connect(DB_PATH)
    cur = conn.cursor()

    all_symbols = [
        row[0]
        for row in cur.execute(
            "SELECT symbol FROM companies ORDER BY symbol"
        ).fetchall()
    ]
    symbols = [s for s in all_symbols if _is_equity_symbol(s)]
    if not symbols:
        conn.close()
        raise ValueError("No equity symbols found in companies table.")

    cur.execute("DELETE FROM price_history")
    conn.commit()

    pending_rows: list[tuple[Any, ...]] = []
    inserted_rows = 0
    failed_symbols: list[str] = []
    fallback_symbols = 0
    github_symbols = 0
    aborted_early = False

    client = Client()
    try:
        nepse_ok, fallback_ok, github_ok = await _preflight_api(client)
        print(
            f"Preflight -> nepse-api: {nepse_ok}, fallback-api: {fallback_ok}, github-open-data: {github_ok}"
        )
        if not nepse_ok and not fallback_ok and not github_ok:
            raise RuntimeError(
                "All data providers are unavailable (nepse-api, fallback API, github-open-data)."
            )

        consecutive_failures = 0
        max_consecutive_failures = 25

        for symbol in tqdm(symbols, desc="Price history", unit="symbol"):
            try:
                history_rows: list[Any] = []
                source_used = ""

                if nepse_ok:
                    try:
                        history_rows = await _get_history_rows_nepse_api(client, symbol)
                        if history_rows:
                            source_used = "nepse"
                    except Exception:
                        history_rows = []

                if not history_rows and fallback_ok:
                    try:
                        history_rows = _get_history_rows_fallback_api(symbol)
                        if history_rows:
                            source_used = "fallback-api"
                    except Exception:
                        history_rows = []

                if not history_rows and github_ok:
                    history_rows = _get_history_rows_github(symbol)
                    if history_rows:
                        source_used = "github-open-data"

                if not history_rows:
                    raise RuntimeError(
                        "No history rows returned from any available provider."
                    )

                if source_used == "fallback-api":
                    fallback_symbols += 1
                elif source_used == "github-open-data":
                    github_symbols += 1

                for item in history_rows:
                    trade_date = _to_iso_date(
                        _first_non_null(
                            item,
                            [
                                "business_date",
                                "businessDate",
                                "trade_date",
                                "tradeDate",
                                "date",
                                "published_date",
                            ],
                        )
                    )
                    if trade_date is None:
                        continue

                    open_price = _to_float(
                        _first_non_null(item, ["open_price", "openPrice", "open"])
                    )
                    high_price = _to_float(
                        _first_non_null(item, ["high_price", "highPrice", "high"])
                    )
                    low_price = _to_float(
                        _first_non_null(item, ["low_price", "lowPrice", "low"])
                    )
                    close_price = _to_float(
                        _first_non_null(
                            item,
                            [
                                "close_price",
                                "closePrice",
                                "last_traded_price",
                                "lastTradedPrice",
                                "close",
                                "ltp",
                            ],
                        )
                    )
                    if close_price is None:
                        continue

                    volume = _to_int(
                        _first_non_null(
                            item,
                            [
                                "total_traded_quantity",
                                "totalTradeQuantity",
                                "total_trade_quantity",
                                "totalTradedQuantity",
                                "volume",
                                "traded_quantity",
                            ],
                        )
                    )
                    turnover = _to_float(
                        _first_non_null(
                            item,
                            [
                                "total_traded_value",
                                "totalTradeValue",
                                "total_trade_value",
                                "turnover",
                                "turn_over",
                                "turnOver",
                                "traded_amount",
                            ],
                        )
                    )
                    if (
                        turnover is None
                        and close_price is not None
                        and volume is not None
                    ):
                        turnover = close_price * volume

                    pending_rows.append(
                        (
                            symbol,
                            trade_date,
                            open_price,
                            high_price,
                            low_price,
                            close_price,
                            volume,
                            turnover,
                        )
                    )

                    if len(pending_rows) >= BATCH_SIZE:
                        cur.executemany(PRICE_INSERT_SQL, pending_rows)
                        conn.commit()
                        inserted_rows += len(pending_rows)
                        pending_rows.clear()

                consecutive_failures = 0
                if source_used in ("nepse", "fallback-api"):
                    time.sleep(SYMBOL_SLEEP_SECONDS)
            except Exception as exc:
                failed_symbols.append(symbol)
                consecutive_failures += 1
                if len(failed_symbols) <= MAX_ERROR_LOGS:
                    print(f"Failed {symbol}: {exc}")

                if consecutive_failures >= max_consecutive_failures:
                    print(
                        "Aborting early: too many consecutive symbol failures. "
                        "This usually means upstream API outage/rate-limit."
                    )
                    aborted_early = True
                    break

        if pending_rows:
            cur.executemany(PRICE_INSERT_SQL, pending_rows)
            conn.commit()
            inserted_rows += len(pending_rows)

        price_row_count = cur.execute("SELECT COUNT(*) FROM price_history").fetchone()[
            0
        ]
        print(f"Equity symbols attempted: {len(symbols)}")
        print(f"Inserted rows (this run): {inserted_rows}")
        print(f"Rows in price_history: {price_row_count}")
        print(f"Symbols via fallback API: {fallback_symbols}")
        print(f"Symbols via github-open-data: {github_symbols}")
        print(f"Failed symbols: {len(failed_symbols)}")
        if failed_symbols:
            print(f"Sample failed symbols: {failed_symbols[:10]}")

        if aborted_early and price_row_count == 0:
            raise RuntimeError(
                "Price ingestion aborted with zero rows due to persistent upstream failures."
            )
    finally:
        await client.close()
        conn.close()


await ingest_price_history()

Preflight -> nepse-api: False, fallback-api: False, github-open-data: True


Price history:   0%|          | 0/338 [00:00<?, ?symbol/s]

Failed ACLBSL: No history rows returned from any available provider.


Price history:   1%|          | 2/338 [00:00<02:25,  2.31symbol/s]

Failed ADLB: No history rows returned from any available provider.
Failed AHL: No history rows returned from any available provider.


...

Failed BBC: No history rows returned from any available provider.
Failed BEDC: No history rows returned from any available provider.


Price history:   5%|▍         | 16/338 [00:08<02:25,  2.21symbol/s]

Failed BGWT: No history rows returned from any available provider.
Failed BHDC: No history rows returned from any available provider.
Failed BHL: No history rows returned from any available provider.
Failed BHPL: No history rows returned from any available provider.
Failed BNHC: No history rows returned from any available provider.
Failed BNL: No history rows returned from any available provider.
Failed BNT: No history rows returned from any available provider.


Price history: 100%|██████████| 338/338 [01:39<00:00,  3.39symbol/s]

Equity symbols attempted: 338
Inserted rows (this run): 235167
Rows in price_history: 235167
Symbols via fallback API: 0
Symbols via github-open-data: 105
Failed symbols: 233
Sample failed symbols: ['ACLBSL', 'ADLB', 'AHL', 'ALBSL', 'ANLB', 'AVYAN', 'BBC', 'BEDC', 'BGWT', 'BHDC']


In [4]:
conn = sqlite3.connect(DB_PATH)
cur = conn.cursor()

summary = cur.execute(
    """
    SELECT
        COUNT(*) AS rows_count,
        COUNT(DISTINCT symbol) AS symbols_count,
        MIN(trade_date) AS min_trade_date,
        MAX(trade_date) AS max_trade_date
    FROM price_history
    """
).fetchone()

print(
    {
        "rows_count": summary[0],
        "symbols_count": summary[1],
        "min_trade_date": summary[2],
        "max_trade_date": summary[3],
    }
)

sample_rows = cur.execute(
    """
    SELECT symbol, trade_date, open_price, high_price, low_price, close_price, volume, turnover
    FROM price_history
    ORDER BY trade_date DESC, symbol
    LIMIT 5
    """
).fetchall()

for row in sample_rows:
    print(row)

conn.close()

{'rows_count': 235167, 'symbols_count': 105, 'min_trade_date': '2003-06-06', 'max_trade_date': '2026-03-29'}
('ADBL', '2026-03-29', 333.0, 335.0, 320.5, 324.0, 106944, 34707703.5)
('AHPC', '2026-03-29', 315.0, 315.0, 303.0, 304.1, 478250, 147221493.0)
('AKJCL', '2026-03-29', 415.0, 420.1, 402.0, 406.9, 565146, 231370717.4)
('AKPL', '2026-03-29', 304.9, 305.0, 291.2, 291.9, 547794, 162204106.3)
('ALICL', '2026-03-29', 490.0, 495.0, 470.0, 470.0, 120921, 57631824.9)


In [5]:
INDICES_INSERT_SQL = """
INSERT INTO indices (
    trade_date, index_name, open_value, close_value, change_points, change_percent, turnover
) VALUES (?, ?, ?, ?, ?, ?, ?)
"""

def _build_index_rows(records: list[Any], forced_name: str) -> list[tuple[Any, ...]]:
    out: list[tuple[Any, ...]] = []
    for rec in records:
        trade_date = _to_iso_date(
            _first_non_null(rec, ["business_date", "businessDate", "trade_date", "tradeDate", "date"])
        )
        close_value = _to_float(
            _first_non_null(
                rec,
                ["close_value", "closeValue", "close_price", "closePrice", "close", "index", "value", "ltp"],
            )
        )
        if trade_date is None or close_value is None:
            continue

        open_value = _to_float(_first_non_null(rec, ["open_value", "openValue", "open_price", "openPrice", "open"]))
        change_points = _to_float(_first_non_null(rec, ["change_points", "changePoints", "point_change", "change"]))
        change_percent = _to_float(
            _first_non_null(rec, ["change_percent", "changePercent", "percentage_change", "percent_change"])
        )
        turnover = _to_float(
            _first_non_null(rec, ["turnover", "turnOver", "total_turnover", "totalTurnover", "turn_over", "turnOverValues"])
        )

        if change_points is None and open_value is not None:
            change_points = close_value - open_value
        if change_percent is None and open_value not in (None, 0):
            change_percent = ((close_value - open_value) / open_value) * 100.0

        out.append(
            (
                trade_date,
                forced_name,
                open_value,
                close_value,
                change_points,
                change_percent,
                turnover,
            )
        )
    return out

def _get_index_rows_github(
    target_days: dict[str, int] | None = None,
    max_files_to_scan: int = 120,
) -> list[tuple[Any, ...]]:
    if target_days is None:
        target_days = {"NEPSE": 300, "Banking": 300, "Hydropower": 50}

    urls = _load_github_index_file_urls()
    seen_dates_by_index: dict[str, set[str]] = {"NEPSE": set(), "Banking": set(), "Hydropower": set()}
    rows: list[tuple[Any, ...]] = []

    scanned_files = 0
    for csv_url in urls:
        if scanned_files >= max_files_to_scan:
            break
        scanned_files += 1

        text = _http_get_text(csv_url)
        reader = csv.DictReader(io.StringIO(text))
        for item in reader:
            canonical_name = _canonical_index_name(item.get("symbol"))
            if canonical_name not in seen_dates_by_index:
                continue

            trade_date = _to_iso_date(item.get("date"))
            if trade_date is None or trade_date in seen_dates_by_index[canonical_name]:
                continue

            open_value = _to_float(item.get("open"))
            close_value = _to_float(item.get("close"))
            if close_value is None:
                continue

            turnover = _to_float(item.get("volume"))
            change_points = None if open_value is None else close_value - open_value
            change_percent = None
            if open_value not in (None, 0):
                change_percent = ((close_value - open_value) / open_value) * 100.0

            rows.append(
                (
                    trade_date,
                    canonical_name,
                    open_value,
                    close_value,
                    change_points,
                    change_percent,
                    turnover,
                )
            )
            seen_dates_by_index[canonical_name].add(trade_date)

        if all(len(seen_dates_by_index[name]) >= target_days.get(name, 0) for name in seen_dates_by_index):
            break

    return rows

async def ingest_indices() -> None:
    conn = sqlite3.connect(DB_PATH)
    cur = conn.cursor()
    cur.execute("DELETE FROM indices")
    conn.commit()

    endpoint_map = {
        "NEPSE": "/DailyNepseIndexGraph",
        "Banking": "/DailyBankSubindexGraph",
        "Hydropower": "/DailyHydroPowerSubindexGraph",
    }

    pending_rows: list[tuple[Any, ...]] = []
    seen_dates_by_index: dict[str, set[str]] = {name: set() for name in endpoint_map}

    # Try online API first for each canonical index.
    for index_name, endpoint in endpoint_map.items():
        try:
            payload = _http_get_json(endpoint)
            records = _normalize_rows(payload)
            rows = _build_index_rows(records, forced_name=index_name)

            for row in rows:
                trade_date = row[0]
                if trade_date in seen_dates_by_index[index_name]:
                    continue
                seen_dates_by_index[index_name].add(trade_date)
                pending_rows.append(row)
        except Exception as exc:
            print(f"Failed {index_name} index via {endpoint}: {exc}")

    # If API is unavailable, use maintained GitHub open-data as deterministic fallback.
    if not pending_rows:
        github_rows = _get_index_rows_github()
        for row in github_rows:
            trade_date, index_name, *_ = row
            if index_name not in seen_dates_by_index:
                continue
            if trade_date in seen_dates_by_index[index_name]:
                continue
            seen_dates_by_index[index_name].add(trade_date)
            pending_rows.append(row)
        print(f"Loaded {len(pending_rows)} index rows from github-open-data fallback.")

    if pending_rows:
        pending_rows.sort(key=lambda r: (r[0], r[1]))
        cur.executemany(INDICES_INSERT_SQL, pending_rows)
        conn.commit()

    missing_targets = [name for name, seen in seen_dates_by_index.items() if not seen]
    inserted = cur.execute("SELECT COUNT(*) FROM indices").fetchone()[0]
    print(f"Rows in indices: {inserted}")
    if missing_targets:
        print(f"WARNING: Missing index history for: {missing_targets}")

    conn.close()

    if inserted == 0:
        raise RuntimeError(
            "Indices ingestion inserted zero rows. Both primary API and fallback source are unavailable."
        )

await ingest_indices()

Failed NEPSE index via /DailyNepseIndexGraph: HTTP fetch failed for https://nepseapi.surajrimal.dev/DailyNepseIndexGraph: HTTP Error 521: <none>
Failed Banking index via /DailyBankSubindexGraph: HTTP fetch failed for https://nepseapi.surajrimal.dev/DailyBankSubindexGraph: HTTP Error 521: <none>
Failed Hydropower index via /DailyHydroPowerSubindexGraph: HTTP fetch failed for https://nepseapi.surajrimal.dev/DailyHydroPowerSubindexGraph: HTTP Error 521: <none>
Loaded 311 index rows from github-open-data fallback.
Rows in indices: 311


In [6]:
conn = sqlite3.connect(DB_PATH)
cur = conn.cursor()

index_counts = cur.execute(
    """
    SELECT index_name, COUNT(*) AS rows_count, MIN(trade_date) AS min_date, MAX(trade_date) AS max_date
    FROM indices
    GROUP BY index_name
    ORDER BY index_name
    """
).fetchall()

print("Canonical index names expected by benchmark Q11:")
print(["NEPSE", "Banking", "Hydropower"])
print("\nRows per index:")
for row in index_counts:
    print(row)

sample = cur.execute(
    """
    SELECT trade_date, index_name, open_value, close_value, change_points, change_percent, turnover
    FROM indices
    ORDER BY trade_date DESC, index_name
    LIMIT 10
    """
).fetchall()

print("\nLatest sample rows:")
for row in sample:
    print(row)

conn.close()

Canonical index names expected by benchmark Q11:
['NEPSE', 'Banking', 'Hydropower']

Rows per index:
('Banking', 120, '2007-04-19', '2007-10-16')
('Hydropower', 71, '2007-07-03', '2007-10-16')
('NEPSE', 120, '2007-04-19', '2007-10-16')

Latest sample rows:
('2007-10-16', 'Banking', 939.6, 939.6, 0.0, 0.0, None)
('2007-10-16', 'Hydropower', 1377.3, 1377.3, 0.0, 0.0, None)
('2007-10-16', 'NEPSE', 863.1, 863.1, 0.0, 0.0, None)
('2007-10-15', 'Banking', 939.1, 939.1, 0.0, 0.0, None)
('2007-10-15', 'Hydropower', 1377.3, 1377.3, 0.0, 0.0, None)
('2007-10-15', 'NEPSE', 869.3, 869.3, 0.0, 0.0, None)
('2007-10-14', 'Banking', 921.3, 921.3, 0.0, 0.0, None)
('2007-10-14', 'Hydropower', 1377.5, 1377.5, 0.0, 0.0, None)
('2007-10-14', 'NEPSE', 883.4, 883.4, 0.0, 0.0, None)
('2007-10-11', 'Banking', 948.6, 948.6, 0.0, 0.0, None)
